In [1]:
#importing necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer

In [2]:
# Loading the Data set
df = pd.read_csv("D:/Git REPO/ML-ML Zoomcamp-September/Chapter-2-Classification/course_lead_scoring.csv")
df.head()

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
0,paid_ads,NaN,1,79450.0,unemployed,south_america,4,0.94,1
1,social_media,retail,1,46992.0,employed,south_america,1,0.80,0
2,events,healthcare,5,78796.0,unemployed,australia,3,0.69,1
3,paid_ads,retail,2,83843.0,NaN,australia,1,0.87,0
4,referral,education,3,85012.0,self_employed,europe,3,0.62,1


In [3]:
# Understanding the data
df.info()
df.describe()
print()
print(len(df))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1462 entries, 0 to 1461
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               1334 non-null   object 
 1   industry                  1328 non-null   object 
 2   number_of_courses_viewed  1462 non-null   int64  
 3   annual_income             1281 non-null   float64
 4   employment_status         1362 non-null   object 
 5   location                  1399 non-null   object 
 6   interaction_count         1462 non-null   int64  
 7   lead_score                1462 non-null   float64
 8   converted                 1462 non-null   int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 102.9+ KB

1462


In [4]:
# Most frequent observation (mode) for the column industry
industry_mode = df['industry'].mode()[0]
print(industry_mode)

retail


In [5]:
# Correlation matrix for numerical features
numerical_cols = ['number_of_courses_viewed', 'annual_income', 'interaction_count', 'lead_score']
correlation_matrix = df[numerical_cols].corr()
print(correlation_matrix)

# Finding the biggest correlation among given pairs
corr_pairs = {
    ('interaction_count', 'lead_score'): correlation_matrix.loc['interaction_count', 'lead_score'],
    ('number_of_courses_viewed', 'lead_score'): correlation_matrix.loc['number_of_courses_viewed', 'lead_score'],
    ('number_of_courses_viewed', 'interaction_count'): correlation_matrix.loc['number_of_courses_viewed', 'interaction_count'],
    ('annual_income', 'interaction_count'): correlation_matrix.loc['annual_income', 'interaction_count']
}
max_corr_pair = max(corr_pairs, key=corr_pairs.get)
print( )
print("The pair with the biggest correlation is:", end=" ")
print( max_corr_pair)

                          number_of_courses_viewed  annual_income  \
number_of_courses_viewed                  1.000000       0.031551   
annual_income                             0.031551       1.000000   
interaction_count                        -0.023565       0.048618   
lead_score                               -0.004879       0.005334   

                          interaction_count  lead_score  
number_of_courses_viewed          -0.023565   -0.004879  
annual_income                      0.048618    0.005334  
interaction_count                  1.000000    0.009888  
lead_score                         0.009888    1.000000  

The pair with the biggest correlation is: ('annual_income', 'interaction_count')


In [6]:
# Split the data
X = df.drop('converted', axis=1)  # Features
y = df['converted']  # Target
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

In [7]:
# Inspect missing values in training set
print("Missing per column in X_train (before imputation):")
print(X_train.isnull().sum())


Missing per column in X_train (before imputation):
lead_source                  80
industry                     92
number_of_courses_viewed      0
annual_income               114
employment_status            59
location                     40
interaction_count             0
lead_score                    0
dtype: int64


In [8]:
# Identify numeric and categorical columns
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = ['lead_source', 'industry', 'employment_status', 'location']


In [9]:
# Impute numeric columns using median (fit on train only)
num_imputer = SimpleImputer(strategy='median')
X_train_num = pd.DataFrame(num_imputer.fit_transform(X_train[numeric_cols]),
                           columns=numeric_cols, index=X_train.index)
X_val_num = pd.DataFrame(num_imputer.transform(X_val[numeric_cols]),
                         columns=numeric_cols, index=X_val.index)
X_test_num = pd.DataFrame(num_imputer.transform(X_test[numeric_cols]),
                          columns=numeric_cols, index=X_test.index)

In [10]:
# Fill categorical missing values using mode computed from X_train
X_train_cat = X_train[categorical_cols].copy()
X_val_cat = X_val[categorical_cols].copy()
X_test_cat = X_test[categorical_cols].copy()

for col in categorical_cols:
    mode_val = X_train_cat[col].mode(dropna=True)
    fill_value = mode_val.iloc[0] if len(mode_val) > 0 else "missing"
    X_train_cat[col] = X_train_cat[col].fillna(fill_value)
    X_val_cat[col]   = X_val_cat[col].fillna(fill_value)
    X_test_cat[col]  = X_test_cat[col].fillna(fill_value)

In [11]:
# Reconstruct X_train/X_val/X_test with imputed numeric + filled categorical
# Keep any other non-categorical non-numeric columns handled above if present
other_cols = [c for c in X_train.columns if c not in numeric_cols + categorical_cols]
X_train_other = X_train[other_cols].copy() if other_cols else pd.DataFrame(index=X_train.index)
X_val_other   = X_val[other_cols].copy()   if other_cols   else pd.DataFrame(index=X_val.index)
X_test_other  = X_test[other_cols].copy()  if other_cols   else pd.DataFrame(index=X_test.index)

X_train_filled = pd.concat([X_train_num, X_train_cat, X_train_other], axis=1)
X_val_filled   = pd.concat([X_val_num,   X_val_cat,   X_val_other],   axis=1)
X_test_filled  = pd.concat([X_test_num,  X_test_cat,  X_test_other],  axis=1)

In [12]:
# 4) One-hot encode categorical variables (use dummy_na=False because we filled NaNs)
X_train_oh = pd.get_dummies(X_train_filled, columns=categorical_cols, dummy_na=False)
X_val_oh   = pd.get_dummies(X_val_filled,   columns=categorical_cols, dummy_na=False)

In [13]:
# 5) Align columns between train and val (and test if needed)
X_train_oh, X_val_oh = X_train_oh.align(X_val_oh, join='left', axis=1, fill_value=0)

# (Optional) align test as well if you'll evaluate later
# X_train_oh, X_test_oh = X_train_oh.align(pd.get_dummies(X_test_filled, columns=categorical_cols, dummy_na=False), join='left', axis=1, fill_value=0)

In [14]:
# Verify no NaNs remain
nan_counts = X_train_oh.isnull().sum()
if nan_counts.any():
    print("Columns with NaNs after processing (train):")
    print(nan_counts[nan_counts > 0])
    raise ValueError("NaNs remain in X_train_oh after imputation.")
else:
    print("No NaNs in X_train_oh. Ready to fit.")


No NaNs in X_train_oh. Ready to fit.


In [16]:
# Train model
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train_oh, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'liblinear'
,max_iter,1000
,multi_class,'deprecated'


In [18]:
# Predict and calculate accuracy
y_val_pred = model.predict(X_val_oh)
accuracy = round(accuracy_score(y_val, y_val_pred), 2)
print("Accuracy on validation set:", accuracy)

Accuracy on validation set: 0.77


In [19]:
original_accuracy = accuracy
feature_differences = {}
for feature in ['industry', 'employment_status', 'lead_score']:
    X_train_reduced = X_train_oh.drop(feature, axis=1, errors='ignore')
    X_val_reduced = X_val_oh.drop(feature, axis=1, errors='ignore')
    model_reduced = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model_reduced.fit(X_train_reduced, y_train)
    y_val_pred_reduced = model_reduced.predict(X_val_reduced)
    accuracy_reduced = accuracy_score(y_val, y_val_pred_reduced)
    difference = original_accuracy - accuracy_reduced
    feature_differences[feature] = difference
    print(f"Difference for {feature}: {difference}")

min_difference_feature = min(feature_differences, key=feature_differences.get)
print("Least useful feature:", min_difference_feature)

Difference for industry: -0.003972602739725994
Difference for employment_status: -0.003972602739725994
Difference for lead_score: 0.0028767123287671836
Least useful feature: industry


In [21]:
# Regularized logistic regression with different C values
c_values = [0.01, 0.1, 1, 10, 100]
best_accuracy = 0
best_c = None
for c in c_values:
    model_c = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model_c.fit(X_train_oh, y_train)
    y_val_pred_c = model_c.predict(X_val_oh)
    accuracy_c = round(accuracy_score(y_val, y_val_pred_c), 3)
    print(f"Accuracy for C={c}: {accuracy_c}")
    if accuracy_c > best_accuracy:
        best_accuracy = accuracy_c
        best_c = c
print("Best C value:", best_c)

Accuracy for C=0.01: 0.781
Accuracy for C=0.1: 0.774
Accuracy for C=1: 0.774
Accuracy for C=10: 0.774
Accuracy for C=100: 0.774
Best C value: 0.01
